[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Feature Importance


## What you will be able to do

Measure how much a fitted model relies on each feature by shuffling it on held-out rows, and tell
when a forest's built-in importances credit a column that does nothing.

## The idea

### The problem

A random forest predicts the stations' January means well, and the first question people in a
meeting ask next is which features matter. A forest has an answer ready, `feature_importances_`,
a number for every column that adds up to 1. It is tempting to read it as a ranking of causes and
report it as found.

It is computed while the trees grow, from the rows they learned from, so a column of pure noise that
the trees happened to split on earns a share of it. And when two columns carry the same information,
each split can use only one of them, so the credit for that information is divided between the two,
and each looks less important than the information it carries.

The network's stations are made up, and the code that makes them is in Setup, so here the true
answer is known. `station_network` computes each January mean from latitude, elevation, distance
from the coast and terrain. The year a station was installed plays no part and is unrelated to every
column that does, so any importance given to it measures the method itself. Longitude plays no part
either, but it rises with latitude, so a model can read latitude from it.

### What permutation importance is

> The **permutation importance** of a feature is how much a fitted model's score drops when that
> feature's values are shuffled among the rows of a dataset, which breaks the link between the
> feature and the target while leaving everything else as it was. For an answer about new data, that
> dataset is a held-out set. scikit-learn's
> **`permutation_importance(model, X, y, n_repeats=..., random_state=...)`** shuffles each column
> `n_repeats` times and reports the mean drop and its standard deviation.

### Why it works that way

Shuffling a column keeps its values, so the column still looks like latitudes or elevations, while
every row now has somebody else's. If the model relied on the column, its predictions get worse, and
the drop in score says by how much. If it did not, the predictions barely move, and the drop is near
zero, sometimes slightly below it by chance.

Measuring on held-out rows matters because a forest has learned its training rows almost by heart.
On those rows it can use a column of noise to recover answers it memorized, and shuffling the noise
then costs it something. On rows it has not seen, the noise was never useful, and its importance
falls to about zero.

The method treats the model as a box that makes predictions, so it works the same way for a forest,
a linear model or anything else with `predict` and `score`. That is also its limit, since it measures
what this model uses and says nothing about what causes the target. Two correlated columns still confuse it, because
shuffling one leaves the other to stand in for it.

### Where this shows up

Permutation importance belongs wherever a model's results are explained to someone, as in a report of
what drives a prediction, a check that a model is not leaning on a column that leaks the answer, or a
decision about which measurements are worth collecting. The **Train and Test** notebook made the held-out set this
method needs.

### What this notebook covers

- a forest's `feature_importances_`, and what it measures
- permutation importance on the test set, with its spread
- the built-in and permutation rankings side by side
- permutation importance for a linear model, and why coefficients are not importances
- correlated features sharing credit, shown by removing one
- four errors, three of them silent

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(0)
X = rng.normal(size=(300, 2))
y = 3 * X[:, 0] + rng.normal(0, 1.0, 300)

forest = RandomForestRegressor(random_state=0).fit(X[:200], y[:200])
print(forest.feature_importances_.round(2))
result = permutation_importance(forest, X[200:], y[200:], n_repeats=10, random_state=0)
print(result.importances_mean.round(3))
```

```
[0.96 0.04]
[ 1.939 -0.005]
```

`y` depends on column 0 alone, and column 1 is noise. The forest's own importances still gave the
noise 0.04. Shuffling each column on the last 100 rows, which the forest did not learn from, cost it
1.939 in R² for column 0 and nothing for column 1, whose -0.005 is chance.

## Setup

Two libraries for the data, scikit-learn, and four of its tools.

- `numpy` and `pandas` build the stations and the tables of importances, and the cell prints
  scikit-learn's version
- `train_test_split` holds a quarter of the stations back for measuring
- `RandomForestRegressor` is the model whose importances are read, and `LinearRegression` the model
  that shows the method works for any model
- `permutation_importance` shuffles each feature on the held-out rows and measures the drop in score

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. Its code
shows which columns the January mean depends on: latitude, elevation, distance from the coast and
terrain. `FEATURES` names the five numeric columns the models use, and two of them, `longitude` and
`installed`, play no part in the formula.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### A forest to explain

The model is fitted on three quarters of the stations, and the rest are held back:

In [2]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

forest = RandomForestRegressor(random_state=0).fit(X_train, y_train)

print("R² on training rows:", round(forest.score(X_train, y_train), 3))
print("R² on test rows:    ", round(forest.score(X_test, y_test), 3))

R² on training rows: 0.983
R² on test rows:     0.894


The forest fits its own rows almost perfectly and does well on the others. The gap between 0.983 and
0.894 is the memorizing that the **Train and Test** notebook showed.

### What feature_importances_ reports

In [3]:
built_in = pd.Series(forest.feature_importances_, index=FEATURES)

print(built_in.sort_values(ascending=False).round(3))
print("sum:", round(built_in.sum(), 3))

latitude       0.479
coast_km       0.248
elevation_m    0.196
longitude      0.059
installed      0.018
dtype: float64
sum: 1.0


Each tree grows by choosing splits that reduce its squared error on the training rows, and a
feature's importance is the reduction from the splits on it, summed over the tree, averaged over
the forest and scaled to add up to 1. Latitude gets almost half. Longitude gets 0.059 and `installed`
0.018, although the January mean does not depend on either. Those shares are real splits the trees
made, measured on rows the trees had learned.

### Permutation importance on the test set

In [4]:
result = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=0)
permuted = pd.DataFrame({"drop in R²": result.importances_mean, "spread": result.importances_std},
                        index=FEATURES)

permuted.sort_values("drop in R²", ascending=False).round(3)

,drop in R²,spread
latitude,0.660,0.036
elevation_m,0.329,0.019
coast_km,0.309,0.024
longitude,0.031,0.003
installed,-0.002,0.001


Shuffling latitude among the 300 test stations cut the forest's R² by 0.66 on average, from 0.894 to
about 0.23, and the ten drops had a standard deviation of 0.036. `installed` came out at -0.002,
which is what a column looks like when the model's splits on it do not help on new stations, since
shuffling it changed nothing beyond chance. `random_state` fixes the shuffles, so the table is the
same on every run. A single shuffle, done by hand, shows
where the numbers come from:

In [5]:
shuffled = X_test.copy()
shuffled["latitude"] = np.random.default_rng(0).permutation(shuffled["latitude"].to_numpy())

print("R² as measured:         ", round(forest.score(X_test, y_test), 3))
print("R² with latitude shuffled:", round(forest.score(shuffled, y_test), 3))

R² as measured:          0.894
R² with latitude shuffled: 0.209


One shuffle took the R² to 0.209, and `permutation_importance` averaged ten of them.

### The built-in and permutation rankings side by side

In [6]:
side_by_side = pd.DataFrame({"built in": built_in, "permutation": result.importances_mean})

side_by_side.sort_values("permutation", ascending=False).round(3)

,built in,permutation
latitude,0.479,0.660
elevation_m,0.196,0.329
coast_km,0.248,0.309
longitude,0.059,0.031
installed,0.018,-0.002


The two methods agree that latitude comes first. The built-in importances put the distance from the
coast ahead of elevation, 0.248 to 0.196. By permutation the two are close, 0.329 and 0.309, a gap
about the size of either spread, and the spread measures only the shuffles, so this split does not
say which comes second. The clearest difference is `installed`, 0.018 built in against -0.002 by
permutation. The built-in numbers describe how the
trees were grown, and the permutation numbers describe how much the predictions on new stations
depend on each column, which is usually the question being asked.

### Any model with a score

In [7]:
linear = LinearRegression().fit(X_train, y_train)
linear_result = permutation_importance(linear, X_test, y_test, n_repeats=10, random_state=0)

print("R² on test rows:", round(linear.score(X_test, y_test), 3))
print(pd.DataFrame({"coefficient": linear.coef_, "drop in R²": linear_result.importances_mean},
                   index=FEATURES).round(4))

R² on test rows: 0.888
             coefficient  drop in R²
latitude         -0.5305      0.8055
longitude        -0.0113      0.0004
elevation_m      -0.0062      0.3982
coast_km         -0.0441      0.2417
installed         0.0044      0.0004


A linear model has no `feature_importances_`, and permutation importance needs none. Its coefficients
are not importances either, because each is in its own feature's units. Elevation's -0.0062 is per
meter and latitude's -0.5305 per degree, and elevation spans hundreds of meters. Shuffling puts them
on one scale, the drop in R², where elevation counts about half as much as latitude.

### Correlated features share credit

Longitude rises with latitude in this network, because Norway runs from southwest to northeast:

In [8]:
print("correlation of latitude and longitude:", round(X["latitude"].corr(X["longitude"]), 2))

correlation of latitude and longitude: 0.97


When latitude is shuffled, the forest can still read much of it from longitude, so latitude's drop
understates how much the predictions depend on where a station is. Fitting the forest again without
longitude shows it:

In [9]:
without = [name for name in FEATURES if name != "longitude"]
forest_without = RandomForestRegressor(random_state=0).fit(X_train[without], y_train)
result_without = permutation_importance(forest_without, X_test[without], y_test, n_repeats=10,
                                        random_state=0)

print("R² on test rows:", round(forest_without.score(X_test[without], y_test), 3))
print(pd.Series(result_without.importances_mean, index=without).round(3))

R² on test rows: 0.887
latitude       0.865
elevation_m    0.331
coast_km       0.304
installed     -0.002
dtype: float64


Without longitude, the test R² hardly moved, from 0.894 to 0.887, and latitude's importance rose from
0.66 to 0.865. Longitude added almost nothing, and a share of latitude's importance had been hiding in
it. `permutation_importance` shuffles one column at a time, so with correlated features, drop one and
measure again, as this section did, and judge the group as a whole.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/11-feature-importance-solutions.ipynb).

**1.** Compute permutation importance for `forest` with `n_repeats=30` and `random_state=1`, and print the mean drop for each feature rounded to three places, sorted from largest to smallest.

In [10]:
# your code here

**2.** Fit `RandomForestRegressor(max_depth=3, random_state=0)` on the training rows, and print its `feature_importances_` beside the deeper forest's, rounded to three places.

In [11]:
# your code here

**3.** Compute permutation importance for `forest` with `scoring="neg_mean_absolute_error"`, and print the mean drops. Each is now how many degrees the mean absolute error grows when the feature is shuffled.

In [12]:
# your code here

**4.** Fit a forest on the training rows without `installed`, and print its R² on the test rows next to the full forest's, rounded to three places.

In [13]:
# your code here

**5.** Fit a forest that predicts `snow_on_may_1` from `FEATURES` with `RandomForestClassifier(random_state=0)`, and print the permutation importance of each feature on a held-out quarter of the stations, with `random_state=0` for both the split and the shuffles.

In [14]:
# your code here

## Common errors

### AttributeError: 'LinearRegression' object has no attribute 'feature_importances_'

In [15]:
linear.feature_importances_

AttributeError: 'LinearRegression' object has no attribute 'feature_importances_'

Only models built from trees have `feature_importances_`. A linear model has `coef_`, whose values are
in each feature's units. `permutation_importance` works for both and puts them on one scale.

### No error, and importance for a column of noise

In [16]:
X_noise = X.assign(noise=np.random.default_rng(1).normal(size=len(X)))
X_noise_train, X_noise_test = train_test_split(X_noise, test_size=0.25, random_state=0)
forest_noise = RandomForestRegressor(random_state=0).fit(X_noise_train, y_train)

noise_built_in = pd.Series(forest_noise.feature_importances_, index=X_noise.columns)
print(noise_built_in.sort_values(ascending=False).round(3))

latitude       0.475
coast_km       0.244
elevation_m    0.192
longitude      0.056
noise          0.018
installed      0.015
dtype: float64


`noise` is random numbers, drawn with a fixed seed and unrelated to anything, and the forest's own
importances gave it 0.018, more than `installed`. A tree that can split on any value of a continuous
column finds splits that reduce the error on its training rows a little, and those count. Measure on
the test rows instead:

In [17]:
noise_result = permutation_importance(forest_noise, X_noise_test, y_test, n_repeats=10, random_state=0)

print(pd.Series(noise_result.importances_mean, index=X_noise.columns).round(3))

latitude       0.653
longitude      0.032
elevation_m    0.325
coast_km       0.303
installed     -0.001
noise          0.000
dtype: float64


On stations the forest did not learn from, shuffling the noise cost nothing.

### No error, and permutation importance measured on the training rows

In [18]:
train_result = permutation_importance(forest_noise, X_noise_train, y_train, n_repeats=10, random_state=0)

print(pd.Series(train_result.importances_mean, index=X_noise.columns).round(3))

latitude       0.790
longitude      0.052
elevation_m    0.313
coast_km       0.406
installed      0.012
noise          0.015
dtype: float64


On the rows it learned, the forest had used the noise to recover answers it memorized, so shuffling
the noise cost it 0.015, and `installed` 0.012. The same call on the test rows gave both about zero.
Permutation importance measures what helps on the rows it is given, so give it rows the model has not
seen.

### No error, and a copy of latitude sharing its importance

In [19]:
X_twice = X.assign(latitude_again=X["latitude"])
X_twice_train, X_twice_test = train_test_split(X_twice, test_size=0.25, random_state=0)
forest_twice = RandomForestRegressor(random_state=0).fit(X_twice_train, y_train)
twice_result = permutation_importance(forest_twice, X_twice_test, y_test, n_repeats=10, random_state=0)

print("R² on test rows:", round(forest_twice.score(X_twice_test, y_test), 3))
print(pd.DataFrame({"built in": forest_twice.feature_importances_,
                    "permutation": twice_result.importances_mean}, index=X_twice.columns).round(3))

R² on test rows: 0.895
                built in  permutation
latitude           0.211        0.143
longitude          0.058        0.030
elevation_m        0.196        0.329
coast_km           0.247        0.310
installed          0.018       -0.001
latitude_again     0.270        0.212


`latitude_again` is the same column under another name. The model scored as well as before, 0.895,
and both methods split latitude's credit between the two copies: 0.211 and 0.27 built in, 0.143 and
0.212 by permutation, where latitude alone had 0.479 and 0.66. Neither copy looks as important as
latitude is, because shuffling one leaves the other. A copy is the extreme case of the correlated
longitude above. Look for columns that measure the same thing before reading importances, and judge
them as a group.

## Recap

- Read `feature_importances_` as a summary of how a forest's trees were grown on their training rows
- Measure permutation importance on held-out rows, with `n_repeats` and `random_state`
- Compare features with permutation importance for any model, where coefficients are in different
  units
- Recognize importance given to noise, and the credit two correlated columns split between them

## What is next

The **Mixed Column Types** notebook builds one pipeline for a table with numbers, text and gaps, the
shape real data arrives in.

---

&#8592; **Previous:** [Hyperparameter Search](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/10-hyperparameter-search.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Mixed Column Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/12-mixed-column-types.ipynb) &#8594;
